In [ ]:

#  POTHOLE SEGMENTATION
import cv2
import os
import numpy as np
 
input_folder = "road2_member2_gaussian_contrast"
 
# Separate folder for each stage
folders_cell3 = {
    "input":     "segmentation_results1/1_input_frames",
    "gray":      "segmentation_results1/2_gray_frames",
    "blur":      "segmentation_results1/3_blur_frames",
    "mask":      "segmentation_results1/4_mask_frames",
    "contours":  "segmentation_results1/5_contour_frames",
    "detected":  "segmentation_results1/6_detected_frames",
}
 
# Create all folders
for folder in folders_cell3.values():
    os.makedirs(folder, exist_ok=True)
 
min_area = 200
max_area = 5000
 
for file in os.listdir(input_folder):
    path = os.path.join(input_folder, file)
    img = cv2.imread(path)
    if img is None:
        continue
 
    # Save input
    cv2.imwrite(os.path.join(folders_cell3["input"], file), img)
 
    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders_cell3["gray"], file), gray)
 
    # Gaussian Blur
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    cv2.imwrite(os.path.join(folders_cell3["blur"], file), blur)
 
    # Threshold mask
    _, mask = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)
    cv2.imwrite(os.path.join(folders_cell3["mask"], file), mask)
 
    # Contour detection
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
 
    marked = img.copy()
    mask_valid = np.zeros_like(mask)
 
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area < area < max_area:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(marked, (x, y), (x+w, y+h), (0, 0, 255), 2)
            cv2.putText(marked, "Pothole", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.rectangle(mask_valid, (x, y), (x+w, y+h), 255, -1)
 
    # Save contour mask and detected output separately
    cv2.imwrite(os.path.join(folders_cell3["contours"], file), mask_valid)
    cv2.imwrite(os.path.join(folders_cell3["detected"], file), marked)
 
    print(f"Cell3 - {file} processed and saved.")
 
print("\nCell 3 completed!")
print("Folder structure:")
for name, path in folders_cell3.items():
    print(f"  {path}/")